# LeetCode 1838 — Frequency of the Most Frequent Element bằng Python

Link đề: https://leetcode.com/problems/frequency-of-the-most-frequent-element/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Greedy, Sliding Window, Sorting, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maxFrequency(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Chứng minh hai nhận xét tham lam: (a) giá trị đích tối ưu luôn là **một phần tử có sẵn**; (b) các phần tử được nâng lên luôn là một **đoạn liên tiếp** ngay dưới đích trong mảng **đã sắp**.
2. Tính chi phí nâng một đoạn `[l..r]` lên `nums[r]` trong `O(1)` bằng **tổng tiền tố**: `nums[r]·(r−l+1) − (pre[r+1] − pre[l])`.
3. Giải bằng ba kỹ thuật cùng nền tảng: **nhị phân trên độ dài**, **nhị phân tìm `l` cho mỗi `r`**, và **cửa sổ trượt** (hai con trỏ) `O(n)` sau khi sắp.
4. Hiểu vì sao cửa sổ trượt hợp lệ: chi phí đoạn **đơn điệu** khi mở rộng sang phải hoặc thu hẹp từ trái.
5. Đối chiếu với vét cạn thật sự (thử mọi đích và mọi cách phân bổ phép tăng) trên mảng nhỏ.

## 2. Đề bài & ví dụ

**Tần suất** của một phần tử là số lần nó xuất hiện trong mảng. Cho mảng `nums` và số nguyên `k`. Mỗi thao tác: chọn một chỉ số và **tăng** phần tử đó thêm 1. Trả về **tần suất lớn nhất có thể** của một phần tử sau **tối đa** `k` thao tác.

**Ví dụ 1:** `nums = [1,2,4], k = 5` → `3`: tăng phần tử đầu 3 lần, phần tử thứ hai 2 lần ⇒ `[4,4,4]`.

**Ví dụ 2:** `nums = [1,4,8,13], k = 5` → `2` (nhiều cách: `[4,4,8,13]`, `[1,8,8,13]`, `[1,4,13,13]`).

**Ví dụ 3:** `nums = [3,9,6], k = 2` → `1`.

**Ràng buộc:** `1 ≤ n ≤ 10⁵`; `1 ≤ nums[i] ≤ 10⁵`; `1 ≤ k ≤ 10⁵`.

In [1]:
import random
import time
import bisect
from itertools import accumulate, product
from collections import Counter
from typing import List

examples = [([1, 2, 4], 5, 3), ([1, 4, 8, 13], 5, 2), ([3, 9, 6], 2, 1)]
for nums, k, exp in examples:
    print(f"nums={nums}, k={k} -> kỳ vọng {exp}")

nums=[1, 2, 4], k=5 -> kỳ vọng 3
nums=[1, 4, 8, 13], k=5 -> kỳ vọng 2
nums=[3, 9, 6], k=2 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Chỉ được **tăng** ⇒ chỉ phần tử **nhỏ hơn hoặc bằng** đích mới đổi thành đích được

Phần tử `x` thành `t` cần `t − x` thao tác, và chỉ khi `x ≤ t`. Không có thao tác giảm.

### 3.2 Tổng tiền tố

`pre[i] = nums[0] + … + nums[i−1]`, `pre[0] = 0` ⇒ tổng đoạn `nums[l..r] = pre[r+1] − pre[l]`. `accumulate(nums, initial=0)` tạo nhanh `pre`.

In [2]:
nums = sorted([1, 4, 8, 13])
pre = list(accumulate(nums, initial=0))
l, r = 1, 2
print("pre =", pre, "| tổng nums[1..2] =", pre[r + 1] - pre[l], "=", sum(nums[l:r + 1]))

pre = [0, 1, 5, 13, 26] | tổng nums[1..2] = 12 = 12


### 3.3 Chi phí nâng một đoạn đã sắp lên phần tử cuối

Với `nums` đã sắp tăng, nâng mọi phần tử trong `[l..r]` lên `nums[r]` tốn `nums[r]·(r − l + 1) − (pre[r+1] − pre[l])`.

In [3]:
def cost(nums_sorted, pre, l, r):
    return nums_sorted[r] * (r - l + 1) - (pre[r + 1] - pre[l])

nums = sorted([1, 2, 4]); pre = list(accumulate(nums, initial=0))
print("nâng [1,2,4] lên 4:", cost(nums, pre, 0, 2), "| nâng [2,4] lên 4:", cost(nums, pre, 1, 2))

nâng [1,2,4] lên 4: 5 | nâng [2,4] lên 4: 2


### 3.4 Hai con trỏ / cửa sổ trượt

Duyệt `r` từ trái sang phải; giữ `l` là đầu cửa sổ. Khi cửa sổ "vi phạm" (chi phí `> k`) thì tăng `l`. Mỗi con trỏ chỉ đi tới, nên tổng `O(n)`.

### 3.5 `bisect` trên dãy đơn điệu tự tính

Với `r` cố định, `cost(l, r)` **giảm** khi `l` tăng ⇒ có thể nhị phân tìm `l` nhỏ nhất có `cost ≤ k`.

## 4. Ý tưởng nền tảng / trực giác

**Nhận xét 1 — đích là phần tử có sẵn.** Giả sử ta đưa một nhóm phần tử về cùng giá trị `t`. Nếu `t` không có trong nhóm ban đầu, hạ `t` xuống bằng phần tử **lớn nhất** của nhóm sẽ giảm chi phí mà không đổi tần suất. Vậy chỉ cần thử `t ∈ nums`.

**Nhận xét 2 — nhóm là đoạn liên tiếp ngay dưới đích.** Với đích `t = nums[r]` (mảng đã sắp), nếu muốn có `m` phần tử bằng `t`, rẻ nhất là chọn `m` phần tử **lớn nhất** `≤ t`, tức đoạn `nums[r−m+1..r]` (phần tử gần `t` nhất tốn ít nhất).

Tính tay **Ví dụ 2** (`[1,4,8,13]`, `k = 5`):

| đích | đoạn | chi phí | ≤ 5? |
|---|---|---|---|
| 4 | [1,4] | 3 | có ⇒ tần suất 2 |
| 8 | [4,8] | 4 | có ⇒ 2 |
| 8 | [1,4,8] | 7+4 = 11 | không |
| 13 | [8,13] | 5 | có ⇒ 2 |
| 13 | [4,8,13] | 9+5 = 14 | không |

⇒ đáp án 2.

In [4]:
nums = sorted([1, 4, 8, 13]); pre = list(accumulate(nums, initial=0)); k = 5
for r in range(len(nums)):
    for l in range(r, -1, -1):
        c = cost(nums, pre, l, r)
        print(f"đích {nums[r]:>2}, đoạn {nums[l:r+1]}: chi phí {c} {'OK' if c <= k else 'vượt'}")
        if c > k: break

đích  1, đoạn [1]: chi phí 0 OK
đích  4, đoạn [4]: chi phí 0 OK
đích  4, đoạn [1, 4]: chi phí 3 OK
đích  8, đoạn [8]: chi phí 0 OK
đích  8, đoạn [4, 8]: chi phí 4 OK
đích  8, đoạn [1, 4, 8]: chi phí 11 vượt
đích 13, đoạn [13]: chi phí 0 OK
đích 13, đoạn [8, 13]: chi phí 5 OK
đích 13, đoạn [4, 8, 13]: chi phí 14 vượt


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn "thật" (không dùng nhận xét nào) — chỉ cho mảng tí hon

Thử mọi cách phân bổ số thao tác `0..k` cho từng phần tử (tổng `≤ k`), lấy tần suất lớn nhất. Đây là định nghĩa thuần túy — dùng làm **đáp án chuẩn** kiểm chứng hai nhận xét.

In [5]:
def freq_true_brute(nums, k):
    n = len(nums); best = 0
    for adds in product(range(k + 1), repeat=n):
        if sum(adds) <= k:
            best = max(best, max(Counter(x + a for x, a in zip(nums, adds)).values()))
    return best

for nums, k, exp in examples[:1] + [([3, 9, 6], 2, 1)]:
    print(nums, k, "->", freq_true_brute(nums, k), "| kỳ vọng", exp)

[1, 2, 4] 5 -> 3 | kỳ vọng 3
[3, 9, 6] 2 -> 1 | kỳ vọng 1


### 5.2 Baseline dùng hai nhận xét: thử mọi đích, nâng dần phần tử gần nhất

Sắp mảng; với mỗi `r` (đích `nums[r]`), đi `l` từ `r` lùi về trái, cộng dồn chi phí `nums[r] − nums[l]`, dừng khi vượt `k`.

In [6]:
def freq_quadratic(nums, k):
    a = sorted(nums); best = 1
    for r in range(len(a)):
        spent = 0
        for l in range(r, -1, -1):
            spent += a[r] - a[l]
            if spent > k:
                break
            best = max(best, r - l + 1)
    return best

for nums, k, exp in examples:
    got = freq_quadratic(nums, k); print(got, "kỳ vọng", exp); assert got == exp

3 kỳ vọng 3
2 kỳ vọng 2
1 kỳ vọng 1


**Độ phức tạp:** `O(n log n + n · L)` với `L` = độ dài đoạn duyệt được — tệ nhất `O(n²)` (ví dụ mọi phần tử bằng nhau: chi phí luôn 0, mỗi `r` đi hết về đầu) ⇒ `5·10⁹` với `n = 10⁵`.

**Nút thắt:** với mỗi `r` ta tính lại chi phí từ đầu; nhưng khi `r` tăng 1, đoạn tối ưu chỉ **dịch** sang phải — có thể tái sử dụng.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight (đơn điệu của chi phí):** với mảng đã sắp, `cost(l, r) = a[r]·(r−l+1) − Σa[l..r]`:
- Cố định `r`, `cost` **giảm** khi `l` tăng (bỏ bớt phần tử xa nhất).
- Cố định `l`, `cost` **tăng** khi `r` tăng (đích cao hơn và thêm phần tử).

⇒ Nếu `[l, r]` vi phạm (`cost > k`) thì `[l, r+1]` cũng vi phạm ⇒ `l` tối ưu cho `r+1` `≥` `l` tối ưu cho `r` ⇒ **cửa sổ trượt**: `l` chỉ tiến.

**Quy tắc cửa sổ:**
- Thêm `a[r]`: `window_sum += a[r]`.
- **Trong khi** `a[r]·(r−l+1) − window_sum > k`: bỏ `a[l]`, `l += 1`.
- Cập nhật `best = max(best, r − l + 1)`.

**Invariant:** sau bước `r`, `[l, r]` là đoạn dài nhất kết thúc tại `r` có chi phí `≤ k`.

Hai cách khác dùng cùng tính đơn điệu: **nhị phân `l` cho mỗi `r`** (`O(n log n)`), và **nhị phân trên độ dài `m`** — "có đoạn độ dài `m` chi phí `≤ k` không?" là tính chất đơn điệu theo `m`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Nhị phân trên độ dài `m` (đáp án)

`ok(m)`: tồn tại `r ≥ m−1` có `cost(r−m+1, r) ≤ k`. Nếu `ok(m)` đúng thì `ok(m−1)` đúng (bỏ phần tử đầu đoạn) ⇒ nhị phân tìm `m` lớn nhất.

In [7]:
def freq_binary_length(nums, k):
    a = sorted(nums); n = len(a); pre = list(accumulate(a, initial=0))
    def ok(m):
        return any(a[r] * m - (pre[r + 1] - pre[r + 1 - m]) <= k for r in range(m - 1, n))
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid): lo = mid
        else: hi = mid - 1
    return lo

print([freq_binary_length(n, k) for n, k, _ in examples])

[3, 2, 1]


**Code nộp LeetCode (Cách 2):**

In [8]:
class SolutionBinaryLength:
    def maxFrequency(self, nums: List[int], k: int) -> int:
        a = sorted(nums)
        n = len(a)
        pre = list(accumulate(a, initial=0))

        def ok(m: int) -> bool:
            for r in range(m - 1, n):
                if a[r] * m - (pre[r + 1] - pre[r + 1 - m]) <= k:
                    return True
            return False

        lo, hi = 1, n
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

print([SolutionBinaryLength().maxFrequency(n, k) for n, k, _ in examples])

[3, 2, 1]


**Độ phức tạp Cách 2:** sắp `O(n log n)` + `O(n)` mỗi lần kiểm tra × `log n` lần ⇒ `O(n log n)`.

### 7.2 Cách 3 — Với mỗi `r`, nhị phân tìm `l` nhỏ nhất

In [9]:
def freq_binary_left(nums, k):
    a = sorted(nums); n = len(a); pre = list(accumulate(a, initial=0)); best = 1
    for r in range(n):
        lo, hi = 0, r                           # tìm l nhỏ nhất có cost(l, r) <= k (cost giảm khi l tăng)
        while lo < hi:
            mid = (lo + hi) // 2
            if a[r] * (r - mid + 1) - (pre[r + 1] - pre[mid]) <= k: hi = mid
            else: lo = mid + 1
        best = max(best, r - lo + 1)
    return best

print([freq_binary_left(n, k) for n, k, _ in examples])

[3, 2, 1]


**Độ phức tạp Cách 3:** `O(n log n)`. Ưu điểm: dễ chứng minh đúng (mỗi `r` độc lập).

### 7.3 Cách 4 — Cửa sổ trượt (hai con trỏ)

In [10]:
def freq_window(nums, k):
    a = sorted(nums); l = 0; window = 0; best = 0
    for r, x in enumerate(a):
        window += x
        while x * (r - l + 1) - window > k:
            window -= a[l]; l += 1
        best = max(best, r - l + 1)
    return best

print([freq_window(n, k) for n, k, _ in examples])

[3, 2, 1]


**Code nộp LeetCode (Cách 4):** xem mục 9. **Độ phức tạp:** sắp `O(n log n)` + cửa sổ `O(n)`.

### 7.4 Cách 5 — Cửa sổ "không co" (độ dài cửa sổ chỉ tăng hoặc giữ nguyên)

Mẹo kinh điển: thay `while` bằng `if` — mỗi bước `r`, nếu vi phạm thì chỉ bỏ **một** phần tử bên trái. Cửa sổ không bao giờ ngắn đi, và độ dài cuối cùng `n − l` chính là đáp án (độ dài lớn nhất từng đạt). Nhanh và gọn.

In [11]:
class SolutionNoShrink:
    def maxFrequency(self, nums: List[int], k: int) -> int:
        a = sorted(nums)
        l = 0
        window = 0
        for r, x in enumerate(a):
            window += x
            if x * (r - l + 1) - window > k:
                window -= a[l]
                l += 1
        return len(a) - l

print([SolutionNoShrink().maxFrequency(n, k) for n, k, _ in examples])

[3, 2, 1]


**Vì sao đúng:** độ dài cửa sổ `r − l + 1` là "độ dài tốt nhất đã thấy". Khi cửa sổ (cùng độ dài) vi phạm, ta trượt nó sang phải giữ nguyên độ dài; khi không vi phạm, nó dài thêm 1 — nghĩa là đã tìm được đoạn dài hơn hợp lệ. Cửa sổ có thể **tạm thời** vi phạm nhưng độ dài của nó luôn bằng đáp án tốt nhất tới thời điểm đó.

**Độ phức tạp Cách 5:** như Cách 4.

## 8. TRACE từng bước

Trace Cách 4 trên **Ví dụ 1** (`a = [1,2,4]`, `k = 5`) và Ví dụ 2:

| r | a[r] | window (tổng) | cửa sổ | chi phí `a[r]·len − tổng` | co? | best |
|---|---|---|---|---|---|---|
| 0 | 1 | 1 | [1] | 0 | không | 1 |
| 1 | 2 | 3 | [1,2] | 1 | không | 2 |
| 2 | 4 | 7 | [1,2,4] | 5 | không | 3 |

In [12]:
def trace_window(nums, k):
    a = sorted(nums); l = 0; window = 0; best = 0
    print("a =", a)
    for r, x in enumerate(a):
        window += x
        while x * (r - l + 1) - window > k:
            print(f"   r={r}: chi phí {x * (r - l + 1) - window} > {k} -> bỏ a[{l}]={a[l]}")
            window -= a[l]; l += 1
        best = max(best, r - l + 1)
        print(f"r={r} (a[r]={x}): cửa sổ {a[l:r+1]}, chi phí {x * (r - l + 1) - window}, best={best}")
    return best

assert trace_window([1, 2, 4], 5) == 3
print()
assert trace_window([1, 4, 8, 13], 5) == 2

a = [1, 2, 4]
r=0 (a[r]=1): cửa sổ [1], chi phí 0, best=1
r=1 (a[r]=2): cửa sổ [1, 2], chi phí 1, best=2
r=2 (a[r]=4): cửa sổ [1, 2, 4], chi phí 5, best=3

a = [1, 4, 8, 13]
r=0 (a[r]=1): cửa sổ [1], chi phí 0, best=1
r=1 (a[r]=4): cửa sổ [1, 4], chi phí 3, best=2
   r=2: chi phí 11 > 5 -> bỏ a[0]=1
r=2 (a[r]=8): cửa sổ [4, 8], chi phí 4, best=2
   r=3: chi phí 14 > 5 -> bỏ a[1]=4
r=3 (a[r]=13): cửa sổ [8, 13], chi phí 5, best=2


Trace Cách 5 (không co) trên Ví dụ 2 — để thấy cửa sổ có lúc vi phạm nhưng độ dài vẫn đúng:

In [13]:
a = [1, 4, 8, 13]; k = 5; l = 0; window = 0
for r, x in enumerate(a):
    window += x
    viol = x * (r - l + 1) - window > k
    if viol:
        window -= a[l]; l += 1
    print(f"r={r}: cửa sổ {a[l:r+1]} (độ dài {r - l + 1}), chi phí hiện tại {x * (r - l + 1) - window}, {'đã trượt' if viol else 'mở rộng'}")
print("đáp án =", len(a) - l)

r=0: cửa sổ [1] (độ dài 1), chi phí hiện tại 0, mở rộng
r=1: cửa sổ [1, 4] (độ dài 2), chi phí hiện tại 3, mở rộng
r=2: cửa sổ [4, 8] (độ dài 2), chi phí hiện tại 4, đã trượt
r=3: cửa sổ [8, 13] (độ dài 2), chi phí hiện tại 5, đã trượt
đáp án = 2


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 4** — sắp + cửa sổ trượt.

In [14]:
class Solution:
    def maxFrequency(self, nums: List[int], k: int) -> int:
        nums = sorted(nums)
        l = 0
        window = 0
        best = 0
        for r, x in enumerate(nums):
            window += x
            while x * (r - l + 1) - window > k:
                window -= nums[l]
                l += 1
            best = max(best, r - l + 1)
        return best

for nums, k, exp in examples:
    got = Solution().maxFrequency(nums, k); print(got, exp); assert got == exp

3 3
2 2
1 1


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đích có sẵn).** Xét một lời giải tối ưu đưa tập `S` về giá trị `t`. Mọi `x ∈ S` có `x ≤ t`. Gọi `x* = max(S)`. Đưa `S` về `x*` thay vì `t` tốn ít hơn `|S|·(t − x*) ≥ 0` thao tác ⇒ vẫn hợp lệ, cùng tần suất. Vậy tồn tại lời giải tối ưu có đích là một phần tử của mảng.

**Bổ đề 2 (đoạn liên tiếp).** Với đích `a[r]` và `m` phần tử, chi phí `Σ (a[r] − x)` nhỏ nhất khi chọn `m` phần tử `x ≤ a[r]` **lớn nhất** — trong mảng sắp chính là `a[r−m+1..r]` (có thể chọn đúng `a[r]` vì hòa giá trị thì chi phí bằng nhau).

⇒ Đáp án = `max { r − l + 1 : cost(l, r) ≤ k }`.

**Cửa sổ trượt đúng** nhờ đơn điệu: gọi `L(r)` là `l` nhỏ nhất với `cost(l, r) ≤ k`. Vì `cost(l, r+1) ≥ cost(l, r)`, nếu `cost(l, r) > k` thì `cost(l, r+1) > k` ⇒ `L(r+1) ≥ L(r)`. Vòng `while` dịch `l` từ `L(r)` tới đúng `L(r+1)` (dừng khi chi phí `≤ k`; luôn dừng vì `cost(r, r) = 0 ≤ k`). ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1a. Vét cạn phân bổ | `O((k+1)^n · n)` | `O(n)` |
| 1b. Thử đích, nâng dần | `O(n²)` tệ nhất | `O(n)` |
| 2. Nhị phân độ dài | `O(n log n)` | `O(n)` |
| 3. Nhị phân `l` cho mỗi `r` | `O(n log n)` | `O(n)` |
| 4. Cửa sổ trượt | `O(n log n)` (sắp) + `O(n)` | `O(n)` (bản sao đã sắp) |
| 5. Cửa sổ không co | như 4 | như 4 |

Giá trị trung gian `a[r]·(r−l+1)` tới `10⁵ · 10⁵ = 10¹⁰` — Python không tràn; C++/Java dùng `long`.

In [15]:
rng = random.Random(1838)
cases = {"ngẫu nhiên": [rng.randint(1, 10 ** 5) for _ in range(10 ** 5)], "toàn bằng nhau": [7] * 10 ** 5}
for label, arr in cases.items():
    for name, f in [("Cách 2", SolutionBinaryLength().maxFrequency), ("Cách 3", freq_binary_left), ("Cách 4", Solution().maxFrequency), ("Cách 5", SolutionNoShrink().maxFrequency)]:
        t = time.perf_counter(); r = f(arr, 10 ** 5); print(f"{label:<15} {name}: {r:>6} ({(time.perf_counter()-t)*1000:.0f} ms)")
t = time.perf_counter(); freq_quadratic([7] * 3000, 10 ** 5); print(f"Cách 1b toàn bằng nhau n=3000: {(time.perf_counter()-t)*1000:.0f} ms")

ngẫu nhiên      Cách 2:    499 (188 ms)
ngẫu nhiên      Cách 3:    499 (221 ms)


ngẫu nhiên      Cách 4:    499 (38 ms)
ngẫu nhiên      Cách 5:    499 (26 ms)
toàn bằng nhau  Cách 2: 100000 (3 ms)
toàn bằng nhau  Cách 3: 100000 (164 ms)
toàn bằng nhau  Cách 4: 100000 (11 ms)
toàn bằng nhau  Cách 5: 100000 (7 ms)


Cách 1b toàn bằng nhau n=3000: 322 ms


## 12. Bộ test

Ba lớp kiểm thử: (1) vét cạn "thật" trên mảng tí hon (`n ≤ 4`, `k ≤ 4`) để kiểm chứng **cả hai nhận xét**; (2) đối chiếu các cách với baseline `O(n²)` trên mảng vừa; (3) test lớn giữa các cách nhanh.

In [16]:
ALL = {"quad": freq_quadratic, "binLen": freq_binary_length, "binLeft": freq_binary_left, "window": freq_window,
       "SolBinLen": SolutionBinaryLength().maxFrequency, "noShrink": SolutionNoShrink().maxFrequency, "submit": Solution().maxFrequency}
def run_all(nums, k, true_brute=False):
    res = {name: f(list(nums), k) for name, f in ALL.items()}
    if true_brute:
        res["trueBrute"] = freq_true_brute(nums, k)
    assert len(set(res.values())) == 1, (nums, k, res)
    return res["submit"]

for nums, k, exp in examples:
    assert run_all(nums, k) == exp
edge = [([5], 1, 1), ([1, 1, 1], 1, 3), ([1, 100000], 99999, 2), ([1, 100000], 99998, 1), ([2, 2, 1, 1], 1, 3), ([9, 1, 9, 1], 16, 4)]
for nums, k, exp in edge:
    assert run_all(nums, k) == exp, (nums, k)
rng = random.Random(18380)
for _ in range(150):
    run_all([rng.randint(1, 6) for _ in range(rng.randint(1, 4))], rng.randint(1, 4), true_brute=True)
for _ in range(400):
    run_all([rng.randint(1, rng.choice([5, 30, 10 ** 5])) for _ in range(rng.randint(1, 40))], rng.randint(1, 60))
for _ in range(5):
    big = [rng.randint(1, 10 ** 5) for _ in range(20000)]; kk = rng.randint(1, 10 ** 5)
    assert len({f(list(big), kk) for f in (freq_window, SolutionNoShrink().maxFrequency, freq_binary_left, Solution().maxFrequency)}) == 1
print("Ví dụ, edge case, 150 test vét cạn thật, 400 test vừa, 5 test lớn: OK")

Ví dụ, edge case, 150 test vét cạn thật, 400 test vừa, 5 test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp

In [17]:
def wrong_no_sort(nums, k):
    a = list(nums); l = 0; window = 0; best = 0
    for r, x in enumerate(a):
        window += x
        while l <= r and x * (r - l + 1) - window > k:
            window -= a[l]; l += 1
        best = max(best, r - l + 1)
    return best

nums, k = [3, 1, 2], 1                               # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_no_sort(nums, k), "| đúng:", Solution().maxFrequency(nums, k), "| brute:", freq_true_brute(nums, k))

sai : 3 | đúng: 2 | brute: 2


**Vì sao sai:** công thức chi phí `a[r]·len − tổng` chỉ đúng khi `a[r]` là **lớn nhất** trong đoạn. Không sắp thì đoạn có thể chứa phần tử lớn hơn đích (chi phí "âm" vô nghĩa). **Sửa:** sắp trước.

### Lỗi 2 — Chọn đích là giá trị **xuất hiện nhiều nhất ban đầu** (mode)

In [18]:
def wrong_mode_target(nums, k):
    a = sorted(nums); t = Counter(a).most_common(1)[0][0]       # SAI: cố định đích là mode
    below = sorted((x for x in a if x <= t), reverse=True)
    spent = cnt = 0
    for x in below:
        if spent + (t - x) > k: break
        spent += t - x; cnt += 1
    return cnt

nums, k = [1, 1, 10, 11, 12], 3                       # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_mode_target(nums, k), "| đúng:", Solution().maxFrequency(nums, k))

sai : 2 | đúng: 3


**Vì sao sai:** giá trị đang phổ biến nhất (`1`, tần suất 2) chưa chắc là đích tốt nhất; nâng `10, 11` lên `12` chỉ tốn 3 ⇒ tần suất 3. Phải thử **mọi** đích `a[r]`. **Sửa:** cửa sổ trượt duyệt mọi `r`.

### Lỗi 3 — Dùng `if` thay vì `while` khi co cửa sổ, nhưng lại lấy `max(best, r − l + 1)` sau mỗi bước

In [19]:
def wrong_if_max(nums, k):
    a = sorted(nums); l = 0; window = 0; best = 0
    for r, x in enumerate(a):
        window += x
        if x * (r - l + 1) - window > k:             # co một bước (cửa sổ có thể còn vi phạm)
            window -= a[l]; l += 1
        best = max(best, r - l + 1)                   # vẫn đúng vì độ dài không tăng khi vi phạm
    return best

print("kết quả:", [wrong_if_max(n, k) for n, k, _ in examples], "| đúng:", [e for _, _, e in examples])
nums, k = [1, 1, 1, 10], 3
print("trường hợp tự dựng:", wrong_if_max(nums, k), "| đúng:", Solution().maxFrequency(nums, k))

kết quả: [3, 2, 1] | đúng: [3, 2, 1]
trường hợp tự dựng: 3 | đúng: 3


**Nhận xét:** đây thực ra **không sai** — chính là Cách 5 viết khác. Nhiều bạn tưởng phải `while` mới đúng; hiểu vì sao `if` cũng đúng (độ dài cửa sổ không bao giờ vượt đáp án) là điểm cộng khi phỏng vấn. Lỗi **thật** là trộn `if` với việc **thu hẹp** biến `best` ngay khi cửa sổ vi phạm, ví dụ đặt `best = r − l + 1` (gán, không lấy max):

In [20]:
def wrong_assign_best(nums, k):
    a = sorted(nums); l = 0; window = 0; best = 0
    for r, x in enumerate(a):
        window += x
        while x * (r - l + 1) - window > k:
            window -= a[l]; l += 1
        best = r - l + 1                                # SAI: phải lấy max
    return best

nums, k = [1, 2, 3, 100], 3                            # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_assign_best(nums, k), "| đúng:", Solution().maxFrequency(nums, k))

sai : 1 | đúng: 3


**Vì sao sai:** với `while`, cửa sổ cuối (đích 100) chỉ còn `[100]` — đáp án tốt nhất `[1,2,3]` đã xuất hiện **trước** đó. **Sửa:** `best = max(best, r − l + 1)`.

### Lỗi 4 — Cộng chi phí bằng cách duyệt lại cửa sổ mỗi bước (đúng nhưng `O(n²)`)

In [21]:
def slow_recompute(nums, k):
    a = sorted(nums); l = 0; best = 0
    for r in range(len(a)):
        while sum(a[r] - a[j] for j in range(l, r + 1)) > k:   # tính lại tổng mỗi lần
            l += 1
        best = max(best, r - l + 1)
    return best

arr = [5] * 3000
for f in (slow_recompute, Solution().maxFrequency):
    t = time.perf_counter(); r = f(arr, 10); print(f"{getattr(f, '__name__', '?'):<15} -> {r} ({(time.perf_counter()-t)*1000:.0f} ms)")

slow_recompute  -> 3000 (144 ms)
maxFrequency    -> 3000 (0 ms)


**Vì sao không tốt:** khi cửa sổ dài (mảng nhiều phần tử bằng nhau), mỗi bước cộng lại `O(n)` ⇒ `O(n²)`. **Sửa:** duy trì `window` (tổng cửa sổ) cộng/trừ dần.

### Lỗi 5 — Chi phí dùng `a[l]` thay vì `a[r]` làm đích

In [22]:
def wrong_target_left(nums, k):
    a = sorted(nums); l = 0; window = 0; best = 0
    for r, x in enumerate(a):
        window += x
        while window - a[l] * (r - l + 1) > k:      # SAI: hạ về phần tử nhỏ nhất (không được giảm)
            window -= a[l]; l += 1
        best = max(best, r - l + 1)
    return best

nums, k = [1, 1, 10], 9                              # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_target_left(nums, k), "| đúng:", Solution().maxFrequency(nums, k), "| brute:", freq_true_brute(nums, k))

sai : 3 | đúng: 2 | brute: 2


**Vì sao sai:** chỉ được **tăng**, nên đích là phần tử **lớn nhất** của đoạn, không phải nhỏ nhất. Code sai tính "hạ 10 xuống 1" tốn 9 ⇒ tưởng được tần suất 3, nhưng thực tế phải nâng hai số 1 lên 10 tốn 18 > 9. **Sửa:** `a[r] * len − window`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa số phần tử bằng nhau sau ≤ k phép **tăng**" ⇒ **sắp + cửa sổ trượt** với chi phí `a[r]·len − sum`.
- Thấy "chỉ được tăng" ⇒ đích là **max** của nhóm; "chỉ được giảm" ⇒ đích là **min**.
- Chi phí đơn điệu theo hai đầu ⇒ hai con trỏ; nếu không chắc, dùng **nhị phân** cho mỗi `r` (dễ chứng minh hơn).
- Cửa sổ "không co" (`if` thay `while`) cho đáp án là `n − l`.
- Chú ý tràn số ở ngôn ngữ khác (`10¹⁰`).

## 15. Biến thể của bài

1. **Được cả tăng và giảm** (chi phí `|x − t|`): đích tối ưu là **trung vị** của đoạn ⇒ cửa sổ + tổng tiền tố quanh trung vị (**2448 / 2602** dạng tương tự, **1703**).
2. **Mỗi phần tử chỉ được tăng tối đa một lần thêm `d`**: khác hẳn — **2009/2779 Maximum Beauty of an Array After Applying Operation** (cửa sổ với `a[r] − a[l] ≤ 2d`).
3. **2968 Apply Operations to Maximize Frequency Score**: được tăng/giảm, tổng chi phí `≤ k`.
4. **1004 Max Consecutive Ones III**, **424 Longest Repeating Character Replacement**: cửa sổ trượt với ngân sách thay đổi.
5. **Trả về giá trị đích** đạt tần suất lớn nhất (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn phân bổ | Định nghĩa thuần | mũ | `O(n)` | Kiểm chứng nhận xét |
| 1b. Thử đích, nâng dần | Hai nhận xét | `O(n²)` | `O(n)` | Baseline |
| 2. Nhị phân độ dài | `ok(m)` đơn điệu | `O(n log n)` | `O(n)` | Khi quen binary search on answer |
| 3. Nhị phân `l` | Mỗi `r` độc lập | `O(n log n)` | `O(n)` | Dễ chứng minh |
| 4. Cửa sổ trượt | `l` chỉ tiến | `O(n log n)` | `O(n)` | **Bản nộp** |
| 5. Cửa sổ không co | `if`, đáp án `n − l` | `O(n log n)` | `O(n)` | Gọn nhất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([1,1,1,1], 1)`; (b) `([1,2,3,4], 3)`; (c) `([1,2,3,4], 6)`; (d) `([10,1,10,1,10], 9)`.

**Bài 2 (tính tay):** Trace cửa sổ trượt cho `nums = [3,1,6,4,5]`, `k = 3` (nhớ sắp trước).

**Bài 3 (điền TODO):** Hoàn thiện `segment_cost(a_sorted, pre, l, r)`.

**Bài 4 (tự viết lại):** Viết `best_target(nums, k)` trả `(tần suất lớn nhất, giá trị đích nhỏ nhất đạt được tần suất đó)`.

**Bài 5 (mini challenge):** Biến thể **được tăng hoặc giảm** (mỗi thao tác ±1): viết `max_freq_both(nums, k)` bằng cửa sổ trượt + trung vị + tổng tiền tố, đối chiếu vét cạn trên mảng nhỏ.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def segment_cost_todo(a_sorted, pre, l, r):
    # TODO: trả a_sorted[r] * (r - l + 1) - (pre[r + 1] - pre[l])
    pass

In [24]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def best_target(nums, k):
    # TODO: cửa sổ trượt; khi r - l + 1 > best thì cập nhật (best, a[r]); bằng best thì giữ đích nhỏ hơn
    pass

def max_freq_both(nums, k):
    # TODO: chi phí đưa đoạn [l..r] về trung vị m: a[m]*(m-l) - sum(l..m-1) + sum(m+1..r) - a[m]*(r-m)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: sắp. Mức 2: cửa sổ. Mức 3: (a) `4`; (b) `3` (`[2,3,4]`: 2+1 = 3); (c) `4` (`[1..4]` lên 4: 3+2+1 = 6); (d) `4` (`[1,10,10,10]`: 9).

**Bài 2.** Mức 1: `a = [1,3,4,5,6]`. Mức 2: `r = 3` (đích 5): `[3,4,5]` chi phí 3 ✓; `[1,3,4,5]` chi phí 7 ✗. Mức 3: `r = 4` (đích 6): `[4,5,6]` chi phí 3 ✓ ⇒ đáp án `3`.

**Bài 3.** Mức 1: đích là `a[r]`. Mức 2: số phần tử `r − l + 1`. Mức 3: trừ tổng đoạn từ `pre`.

**Bài 4.** Mức 1: thêm biến `target`. Mức 2: cửa sổ dài hơn ⇒ cập nhật cả hai. Mức 3: vì `a[r]` tăng dần, lần đầu đạt độ dài tốt nhất cho đích nhỏ nhất.

**Bài 5.** Mức 1: đích tối ưu là trung vị `a[(l+r)//2]`. Mức 2: chi phí giảm khi bỏ phần tử đầu (đơn điệu) ⇒ vẫn trượt được. Mức 3: dùng `pre` tính hai nửa quanh trung vị; vét cạn: thử mọi đích `t` trong `[min, max]`, chọn các phần tử gần `t` nhất.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1, 2, 3
b1 = [Solution().maxFrequency(n, k) for n, k in [([1, 1, 1, 1], 1), ([1, 2, 3, 4], 3), ([1, 2, 3, 4], 6), ([10, 1, 10, 1, 10], 9)]]
print("Bài 1:", b1); assert b1 == [4, 3, 4, 4]
print("Bài 2:"); assert trace_window([3, 1, 6, 4, 5], 3) == 3
def segment_cost_todo(a_sorted, pre, l, r):
    return a_sorted[r] * (r - l + 1) - (pre[r + 1] - pre[l])
a = [1, 3, 4, 5, 6]; pre = list(accumulate(a, initial=0))
assert all(segment_cost_todo(a, pre, l, r) == sum(a[r] - a[j] for j in range(l, r + 1)) for r in range(5) for l in range(r + 1))
print("Bài 3: OK")

Bài 1: [4, 3, 4, 4]
Bài 2:
a = [1, 3, 4, 5, 6]
r=0 (a[r]=1): cửa sổ [1], chi phí 0, best=1
r=1 (a[r]=3): cửa sổ [1, 3], chi phí 2, best=2
   r=2: chi phí 4 > 3 -> bỏ a[0]=1
r=2 (a[r]=4): cửa sổ [3, 4], chi phí 1, best=2
r=3 (a[r]=5): cửa sổ [3, 4, 5], chi phí 3, best=3
   r=4: chi phí 6 > 3 -> bỏ a[1]=3
r=4 (a[r]=6): cửa sổ [4, 5, 6], chi phí 3, best=3
Bài 3: OK


In [26]:
# Bài 4, 5
def best_target(nums, k):
    a = sorted(nums); l = 0; window = 0; best = 0; target = None
    for r, x in enumerate(a):
        window += x
        while x * (r - l + 1) - window > k:
            window -= a[l]; l += 1
        if r - l + 1 > best:
            best, target = r - l + 1, x
    return best, target

def max_freq_both(nums, k):
    a = sorted(nums); pre = list(accumulate(a, initial=0)); l = 0; best = 0
    def c(l, r):
        m = (l + r) // 2
        return a[m] * (m - l) - (pre[m] - pre[l]) + (pre[r + 1] - pre[m + 1]) - a[m] * (r - m)
    for r in range(len(a)):
        while c(l, r) > k:
            l += 1
        best = max(best, r - l + 1)
    return best

def both_brute(nums, k):
    best = 0
    for t in range(min(nums), max(nums) + 1):
        costs = sorted(abs(x - t) for x in nums); s = 0; cnt = 0
        for cst in costs:
            if s + cst > k: break
            s += cst; cnt += 1
        best = max(best, cnt)
    return best

rng = random.Random(5)
for _ in range(400):
    nums = [rng.randint(1, 15) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, 15)
    bt = best_target(nums, k)
    a = sorted(nums)
    cand = [a[r] for r in range(len(a)) if sum(sorted(a[r] - x for x in a if x <= a[r])[:bt[0]]) <= k and sum(1 for x in a if x <= a[r]) >= bt[0]]
    assert bt[0] == Solution().maxFrequency(nums, k) and bt[1] == min(cand)
    assert max_freq_both(nums, k) == both_brute(nums, k), (nums, k)
print("Bài 4, 5: OK |", best_target([1, 4, 8, 13], 5), max_freq_both([1, 4, 8, 13], 5))

Bài 4, 5: OK | (2, 4) 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sắp + cửa sổ trượt với chi phí nâng lên max":**

```python
a = sorted(nums); l = window = best = 0
for r, x in enumerate(a):
    window += x
    while x * (r - l + 1) - window > k:
        window -= a[l]; l += 1
    best = max(best, r - l + 1)
return best
```

**Checklist:**
- [ ] Đã **sắp** trước khi trượt?
- [ ] Đích là phần tử **lớn nhất** của cửa sổ (`a[r]`), vì chỉ được tăng?
- [ ] Duy trì tổng cửa sổ (không tính lại)?
- [ ] `best = max(...)`, không gán đè?
- [ ] Đã test: mọi phần tử bằng nhau, `n = 1`, khoảng cách lớn (`1` và `10⁵`), `k` vừa đủ / thiếu 1?